# S2阶段：数据预处理、LLM应用和特征工程

本notebook对应S2阶段，主要任务：
- 加载S1阶段处理好的清洗数据
- 筛选加利福尼亚州（CALIFORNIA）的数据作为特征州
- 应用LLM进行文本分析（S2.2路径）
- 特征工程和特征提取
- 为后续的关联规则挖掘（S2.1）和预测建模准备数据


In [1]:
# 导入必要的库
import pandas as pd
import numpy as np
import sys
import os

# 添加src目录到路径
sys.path.append(os.path.join('..', 'src'))

from data_loader import load_fra_reair_data

print("库导入成功！")


库导入成功！


In [2]:
# 加载S1阶段处理好的清洗数据
print("=" * 80)
print("加载FRA REAIR清洗数据...")
print("=" * 80)

# 数据文件路径
data_path = r"../data/processed/df_selected_features_cleaned_new.csv"

# 使用data_loader模块加载数据
df_full = load_fra_reair_data(data_path)

print(f"\n完整数据集形状: {df_full.shape}")
print(f"列数: {df_full.shape[1]}, 行数: {df_full.shape[0]}")
print("\n数据加载完成！")


INFO:data_loader:开始加载FRA REAIR数据...


加载FRA REAIR清洗数据...


INFO:data_loader:FRA REAIR数据加载完成，形状: (222956, 36)
INFO:data_loader:列数: 36, 行数: 222956



完整数据集形状: (222956, 36)
列数: 36, 行数: 222956

数据加载完成！


In [3]:
# 筛选加利福尼亚州（CALIFORNIA）的数据
print("=" * 80)
print("筛选加利福尼亚州数据...")
print("=" * 80)

# 先检查"State Name"列是否存在
if 'State Name' in df_full.columns:
    print(f"✓ 找到'State Name'列")
    
    # 查看State Name列的唯一值（前20个）
    print(f"\nState Name列的唯一值数量: {df_full['State Name'].nunique()}")
    print(f"State Name列的样本值: {df_full['State Name'].unique()[:10]}")
    
    # 筛选加州数据
    california_data = df_full[df_full['State Name'] == 'CALIFORNIA'].copy()
    
    print(f"\n筛选完成！")
    print(f"加州数据行数: {california_data.shape[0]}")
    print(f"占总数据的比例: {california_data.shape[0] / df_full.shape[0] * 100:.2f}%")
else:
    print("⚠ 未找到'State Name'列")
    print("\n可用的列名（前30个）:")
    for i, col in enumerate(df_full.columns[:30], 1):
        print(f"{i:2d}. {col}")


筛选加利福尼亚州数据...
✓ 找到'State Name'列

State Name列的唯一值数量: 50
State Name列的样本值: ['PENNSYLVANIA' 'INDIANA' 'OHIO' 'LOUISIANA' 'MISSISSIPPI' 'FLORIDA'
 'OKLAHOMA' 'GEORGIA' 'NEBRASKA' 'NEW YORK']

筛选完成！
加州数据行数: 11374
占总数据的比例: 5.10%


In [4]:
# 显示加州数据的前5行和形状
print("=" * 80)
print("加州数据预览:")
print("=" * 80)

print(f"\n加州数据形状: {california_data.shape}")
print(f"行数: {california_data.shape[0]}, 列数: {california_data.shape[1]}")

print(f"\n前5行数据:")
print(california_data.head())


加州数据预览:

加州数据形状: (11374, 36)
行数: 11374, 列数: 36

前5行数据:
           Date      Time  Accident Type  Temperature Visibility  \
12   10/18/1996  10:25 AM     Derailment           50        Day   
53   01/09/1995  12:00 PM     Derailment           45        Day   
379  02/13/2016  10:30 AM     Derailment           66        Day   
504  08/21/2020   5:00 AM     Derailment           65       Dawn   
516  12/21/2010   4:50 AM  Other impacts           41       Dark   

    Weather Condition  Equipment Type Equipment Attended  Train Speed  \
12             Cloudy   Freight Train                 No         25.0   
53               Rain   Freight Train                Yes          5.0   
379             Clear  Yard/switching                Yes          6.0   
504             Clear  Yard/switching                Yes          9.0   
516             Clear         UNKNOWN            UNKNOWN          0.0   

    Recorded Estimated Speed  ... Recorded Estimated Speed_clean  \
12                  Recorded 

In [5]:
# 保存加州数据到processed目录
print("=" * 80)
print("保存加州数据...")
print("=" * 80)

# 保存加州数据
california_output_path = "../data/processed/california_data.csv"
california_data.to_csv(california_output_path, index=False, encoding='utf-8-sig')
print(f"✓ 加州数据已保存到: {california_output_path}")

# 创建数据摘要
summary_path = "../data/processed/california_data_summary.txt"
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write("加利福尼亚州数据摘要 (S2阶段特征州)\n")
    f.write("=" * 50 + "\n")
    f.write(f"数据形状: {california_data.shape}\n")
    f.write(f"行数: {california_data.shape[0]}, 列数: {california_data.shape[1]}\n")
    f.write(f"占完整数据集比例: {california_data.shape[0] / df_full.shape[0] * 100:.2f}%\n")
    f.write(f"\n数据来源: data/processed/fra_reair_data_cleaned.csv\n")
    f.write(f"筛选条件: State Name == 'CALIFORNIA'\n")
    
print(f"✓ 数据摘要已保存到: {summary_path}")
print("\n加州数据准备完成！准备进入S2.1和S2.2分析路径。")


保存加州数据...
✓ 加州数据已保存到: ../data/processed/california_data.csv
✓ 数据摘要已保存到: ../data/processed/california_data_summary.txt

加州数据准备完成！准备进入S2.1和S2.2分析路径。
